# P2L maintenance catalog audit

This notebook validates the source coverage, reconciliation policy, and final database-ready catalog. It is an auditable companion to the clean workbook and database refresh.

In [1]:
from pathlib import Path
import json, collections
root = Path.cwd().parents[1]
raw = json.loads((root / 'tmp/maintenance_cleaning/raw_sources.json').read_text(encoding='utf-8'))
clean = json.loads((root / 'tmp/maintenance_cleaning/clean_catalog.json').read_text(encoding='utf-8'))
materials = clean['materials']
providers = clean['providers']
assets = clean['equipment_assets']
audit = clean['source_audit']
clean['quality']

{'generated_at': '2026-08-08T00:26:46.890839+00:00', 'source_file_count': 27, 'catalog_row_count': 123, 'provider_count': 10, 'equipment_asset_count': 3, 'rows_by_type': {'consumable': 43, 'equipment': 23, 'other': 14, 'spare_part': 43}, 'rows_with_part_number': 88, 'rows_with_suggested_stock': 117, 'rows_with_current_stock': 0, 'rows_with_provider': 104, 'rows_with_latest_price': 84, 'duplicate_normalized_keys': 0, 'null_or_blank_names': 0, 'policy': {'stock': 'Planning/list quantities populate suggested_stock only; current_stock remains null unless an actual stock count exists.', 'deduplication': 'Part number after punctuation removal and numeric leading-zero normalization; name slug when no part number exists.', 'pricing': 'Latest 2024 RFL/CMC/SC quotation wins; historical invoices and quotations remain evidence.', 'sources': 'CSV exports duplicated by XLSX were processed but not counted twice; brochures/papers were context only.'}}

## Reconciliation rules

- XLSX files are authoritative over duplicate CSV exports.
- The consolidated main list and laboratory list set suggested quantities.
- Quotation quantities are planning evidence, not physical stock counts.
- Current stock remains blank because no source is a verified stock count.
- The latest 2024 RFL/CMC/SC quotation supplies current quoted prices.
- Part numbers are normalized for punctuation and numeric leading-zero variants.

In [2]:
actual_sources = [f for f in raw['files'] if not Path(f['relative_path']).name.startswith('._') and Path(f['relative_path']).name != '.DS_Store']
assert len(actual_sources) == 27
assert len(audit) == 27
assert {f['relative_path'] for f in actual_sources} == {r['source_file'] for r in audit}
[(r['source_file'], r['classification']) for r in audit]

[('Itens para orcamento.csv', 'duplicate CSV export'), ('Itens para orcamento.xlsx', 'reference or superseded source'), ('Lista de compra mar 2023.csv', 'duplicate CSV export'), ('Lista de compra mar 2023.xlsx', 'reference or superseded source'), ('Lista de compra nov 2022.csv', 'duplicate CSV export'), ('Lista de compra nov 2022.xlsx', 'reference or superseded source'), ('Lista material laboratorio P2L.csv', 'inventory/quotation evidence'), ('Material lab 10 2023.csv', 'duplicate CSV export'), ('Material lab 10 2023.xlsx', 'reference or superseded source'), ('Petrobras project material quotation.xlsx', 'reference or superseded source'), ('SENS administration/1299.pdf', 'inventory/quotation evidence'), ('SENS administration/1376.pdf', 'inventory/quotation evidence'), ('SENS administration/20.074-I - USP LESTE.pdf', 'inventory/quotation evidence'), ('SENS administration/21.310-I USP LESTE atualizado.pdf', 'inventory/quotation evidence'), ('SENS administration/BR-30333-IRMS-253-Plus-BR30

In [3]:
keys = [r['normalized_key'] for r in materials]
assert len(keys) == len(set(keys))
assert all(r['name'].strip() for r in materials)
assert all(r['current_stock'] is None for r in materials)
assert all(r['currency'] in {'BRL', 'EUR', 'USD'} for r in materials)
summary = {
    'materials': len(materials),
    'providers': len(providers),
    'assets': len(assets),
    'by_type': dict(collections.Counter(r['material_type'] for r in materials)),
    'with_part_number': sum(bool(r['part_number']) for r in materials),
    'with_suggested_stock': sum(r['suggested_stock'] is not None for r in materials),
    'with_price': sum(r['unit_cost'] is not None for r in materials),
    'with_provider': sum(bool(r['provider_name']) for r in materials),
}
summary

{'materials': 123, 'providers': 10, 'assets': 3, 'by_type': {'consumable': 43, 'equipment': 23, 'other': 14, 'spare_part': 43}, 'with_part_number': 88, 'with_suggested_stock': 117, 'with_price': 84, 'with_provider': 104}

In [4]:
assert summary['materials'] == 123
assert summary['by_type'] == {'consumable': 43, 'equipment': 23, 'other': 14, 'spare_part': 43}
assert len(providers) == 10
assert len(assets) == 3
'All catalog and coverage checks passed.'

'All catalog and coverage checks passed.'